## 从 SDE 到 Probability Flow ODE

### 1. 核心概念：随机变量与概率分布

- **普通变量 ($x$)**：数轴上的一个确定的“点”。
- **随机变量 ($X$)**：一个“取值的规则”或“可能性的地形图”。
- **概率密度 $p(x)$**：地形图在某个点的高度（拥挤程度）。
- **概率分布**：地形图的整体形状。

#### 核心逻辑：
SDE 虽然写的是 $dx$（个体的位移），但其本质是描述随机变量 $X$ 背后概率分布的演化。

---

### 2. 前向过程：SDE (随机微分方程)

前向加噪过程定义为一个受干扰的动力学系统：
$$
dx = f(x, t)dt + g(t)dw
$$

- **漂移项 $f(x, t)dt$**：确定的“力”，通常将数据拉向原点。
- **扩散项 $g(t)dw$**：随机的“噪声”，负责打碎数据结构，增加系统熵。

#### 微观视角：
每一个样本点 $x$ 都在做剧烈的随机跳动。

#### 宏观视角：
概率分布 $p(x,t)$ 逐渐变平滑，最终被“稀释”为高斯分布。

---

### 3. 宏观演化：Fokker-Planck 方程

虽然 SDE 没写 $p(x,t)$，但根据伊藤引理可推导出密度演化的确定性 PDE：
$$
\frac{\partial p(x, t)}{\partial t} = \underbrace{-\nabla \cdot [f(x, t) p(x, t)]}_{\text{漂移引起的位移}} + \underbrace{\frac{1}{2} g(t)^2 \nabla^2 p(x, t)}_{\text{扩散引起的稀释}}
$$

---

### 4. 关键推导：Probability Flow ODE

#### 目标：
寻找一个确定的 ODE $\frac{dx}{dt} = v(x,t)$，使其密度演化与上述 SDE 完全一致。

#### 推导步骤：
1. **连续性方程**：确定性流动的密度变化规律为 $\frac{\partial p}{\partial t} = -\nabla \cdot (p \cdot v)$。
2. **算子变换**：利用 Score Function $\nabla \ln p = \frac{\nabla p}{p}$，将二阶扩散项降阶：
   $$
   \frac{1}{2} g^2 \nabla^2 p = \frac{1}{2} g^2 \nabla \cdot (p \nabla \ln p)
   $$
3. **合并方程**：
   $$
   \frac{\partial p}{\partial t} = -\nabla \cdot \left[ p \left( f - \frac{1}{2}g^2 \nabla \ln p \right) \right]
   $$
4. **提取速度场**：对比发现 $v = f - \frac{1}{2}g^2 \nabla \ln p$。

#### 结论：
确定性反向公式
$$
dx = \left[ f(x, t) - \frac{1}{2} g(t)^2 \nabla_x \ln p_t(x) \right] dt
$$

---

### 5. 逆向 SDE 与 ODE 的对比

在生成（去噪）阶段，我们需要从 $t=T$ 变回 $t=0$：

| 特性                | 逆向 SDE (Reverse SDE)                                   | 概率流 ODE (PF-ODE)                              |
|---------------------|----------------------------------------------------------|-------------------------------------------------|
| **公式**           | $dx = [f - g^2 \nabla \ln p]dt + g dw$                  | $dx = [f - \frac{1}{2}g^2 \nabla \ln p]dt$    |
| **随机性**         | 保留。每次生成结果不同。                                | 消除。输入噪声确定，结果即唯一。               |
| **系数**           | 系数 1：抵消扩散并为反向噪声腾位。                      | 系数 1/2：仅需精确抵消前向扩散。               |
| **求解器**         | 需要小步长的随机积分（慢）。                            | 可使用 DPM-Solver 高阶加速（快）。             |

---

### 6. 为什么 DPM-Solver 需要 ODE？

- **解析性**：ODE 的解可以被建模为“半线性”结构，DPM-Solver 利用指数积分解决了线性部分，剩下的非线性部分由神经网络拟合。
- **高效性**：因为没有 $dw$ 的随机干扰，求解器可以大胆地跨大步长，在 10-20 步内收敛。
- **一致性**：对于自动驾驶（如 nuPlan 任务），ODE 保证了在相同环境（潜在空间）下规划轨迹的稳定性。

#### 注意：DPM-Solver 的标准推导要求 $f(x, t)$ 必须是关于 $x$ 的线性函数，通常写成 $f(x, t) = f(t)x$。

## 扩散模型前向过程完整推导

### 1. 核心模型：线性漂移 SDE

扩散模型的前向加噪过程被建模为一个 Itô 随机微分方程 (SDE)：
$$
dx = f(t)x dt + g(t) d\mathbf{w}
$$

- $x(t)$：$t \in [0, 1]$ 时刻的数据状态。
- $f(t)x$：漂移项 (Drift)。由于是 $x$ 的线性函数，保证了分布的高斯性。
- $g(t)$：扩散项 (Diffusion)。控制单位时间内注入噪声的强度。
- $d\mathbf{w}$：标准维纳过程（白噪声的积分）。

---

### 2. 解析解推导：积分因子法

为了求得 $x(t)$ 关于 $x(0)$ 的显式表达，我们定义信号缩放系数 $\alpha_t$：
$$
\alpha_t = \exp \left( \int_0^t f(s) ds \right)
$$

#### 推导步骤：
1. **变换变量**：令 $Y(t) = \frac{x(t)}{\alpha_t}$。
2. **求微分**：根据 Itô 公式，$d\left( \frac{x(t)}{\alpha_t} \right) = \frac{g(t)}{\alpha_t} d\mathbf{w}$。（漂移项在此过程中被相互抵消）。
3. **积分**：在 $[0, t]$ 区间求积分，并利用 $\alpha_0 = 1$：
   $$
   \frac{x(t)}{\alpha_t} = x(0) + \int_0^t \frac{g(s)}{\alpha_s} d\mathbf{w}(s)
   $$
4. **复原 $x(t)$**：
   $$
   x(t) = \alpha_t x(0) + \alpha_t \int_0^t \frac{g(s)}{\alpha_s} d\mathbf{w}(s)
   $$
5. 计算分布属性右边第二项是一个对维纳过程的 Itô 积分。  
   根据 Itô Isometry (Itô 等距性)：  
   均值：  
   $\mathbb{E}\left[ \int \dots d\mathbf{w} \right] = 0$。  
   方差：记为 $\sigma_t^2$，其计算公式为：  
   $$\sigma_t^2 = \text{Var}\left( \alpha_t \int_0^t \frac{g(s)}{\alpha_s} d\mathbf{w}(s) \right) = \alpha_t^2 \int_0^t \frac{g(s)^2}{\alpha_s^2} ds$$
---

### 3. 统计特性与参数映射

上述解析解表明 $x(t)$ 是一个高斯过程。我们可以直接写出它的边缘分布概率密度：
$$
q(x(t)|x(0)) = \mathcal{N}(x(t) | \alpha_t x(0), \sigma_t^2 \mathbf{I})
$$

其中，两个关键参数与 SDE 系数 $f, g$ 的映射关系为：

1. **均值系数（信号残余）**
   $$
   \alpha_t = \exp \left( \int_0^t f(s) ds \right) \iff f(t) = \frac{d \log \alpha_t}{dt}
   $$
2. **方差系数（累积噪声）**
   根据 Itô Isometry (Itô 等距性)，随机积分项的方差为：
   $$
   \sigma_t^2 = \alpha_t^2 \int_0^t \frac{g(s)^2}{\alpha_s^2} ds \iff g^2(t) = \frac{d\sigma_t^2}{dt} - 2 f(t) \sigma_t^2
   $$

---

### 4. 针对本文 VPSDE 的特化实现

论文采用 Variance Preserving (VP) 调度，其核心目标是让总方差始终为 $1$。

| 参数名称       | 符号表示       | 与噪声计划 $\beta(t)$ 的关系                          |
|----------------|----------------|------------------------------------------------------|
| **漂移项系数** | $f(t)$         | $-\frac{1}{2}\beta(t)$                              |
| **扩散项系数** | $g^2(t)$       | $\beta(t)$                                          |
| **信号缩放**   | $\alpha_t$    | $\exp \left( -\frac{1}{2} \int_0^t \beta(s) ds \right)$ |
| **噪声强度**   | $\sigma_t^2$  | $1 - \alpha_t^2$                                    |

#### 结论：
在 VPSDE 中，由于满足 $\alpha_t^2 + \sigma_t^2 = 1$，数据在加噪过程中始终保持在单位方差的超球面上。

---

### 5. 总结：为什么能“一步加噪”？

通过上述推导，我们将复杂的 SDE 轨迹演化浓缩为了一个简单的重参数化公式：
$$
   x(t) = \alpha_t x(0) + \sigma_t \epsilon, \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})
$$

- **数学本质**：线性 SDE 的解是封闭的高斯分布。
- **训练意义**：不需要模拟中间步骤，给定任意 $t$ 即可通过 $x(0)$ 瞬移到 $x(t)$ 进行训练。
- **推理意义**：该公式定义的 $\alpha_t$ 和 $\sigma_t$ 的导数，直接决定了 DPM-Solver 进行反向去噪时的步长和方向。

## Score Function、噪声与原始数据预测的等价性

在数学上，Score Function ($\nabla \ln p_t$)、噪声 ($\epsilon$) 以及原始数据预测 ($x_0$) 实际上是同一个物理量的三种不同“马甲”。它们之间的等价性建立在扩散模型最基础的前向加噪公式之上：
$$
\mathbf{x}_t = \alpha_t \mathbf{x}_0 + \sigma_t \mathbf{\epsilon}, \quad \mathbf{\epsilon} \sim \mathcal{N}(0, \mathbf{I})
$$

### 1. 从 Score Function ($\nabla \ln p_t$) 到 噪声 ($\epsilon$)

根据高斯分布的性质，当 $\mathbf{x}_t$ 给定时，它的对数概率梯度（即 Score）可以解析地表达。对于高斯分布 $p(\mathbf{x}_t | \mathbf{x}_0) = \mathcal{N}(\alpha_t \mathbf{x}_0, \sigma_t^2 \mathbf{I})$，其 Score 的定义为：
$$
\nabla_{\mathbf{x}_t} \ln p(\mathbf{x}_t | \mathbf{x}_0) = \nabla_{\mathbf{x}_t} \left[ -\frac{1}{2} \left\| \frac{\mathbf{x}_t - \alpha_t \mathbf{x}_0}{\sigma_t} \right\|^2 + \text{const} \right]
$$

利用链式法则求导：
$$
\nabla_{\mathbf{x}_t} \ln p(\mathbf{x}_t | \mathbf{x}_0) = -\frac{\mathbf{x}_t - \alpha_t \mathbf{x}_0}{\sigma_t^2}
$$

注意到前向公式中 $\mathbf{x}_t - \alpha_t \mathbf{x}_0 = \sigma_t \mathbf{\epsilon}$，代入上式得到最重要的等价关系：
$$
\nabla_{\mathbf{x}_t} \ln p_t(\mathbf{x}_t) = - \frac{\mathbf{\epsilon}_\theta(\mathbf{x}_t, t)}{\sigma_t}
$$

#### 物理直觉：
Score 指向概率增加最快的方向，而噪声 $\epsilon$ 是把数据“弄乱”的方向。所以 Score 刚好就是噪声的反方向（负号），并经过标准差 $\sigma_t$ 的缩放。

---

### 2. 从 噪声 ($\epsilon$) 到 原始数据 ($x_0$)

如果我们已经通过神经网络预测出了噪声 $\mathbf{\epsilon}_\theta$，我们就可以通过简单的代数变换，“反推”出如果没有噪声时 $\mathbf{x}_0$ 应该长什么样。

直接从前向公式 $\mathbf{x}_t = \alpha_t \mathbf{x}_0 + \sigma_t \mathbf{\epsilon}$ 移项：
$$
\alpha_t \mathbf{x}_0 = \mathbf{x}_t - \sigma_t \mathbf{\epsilon}_\theta
$$

解出 $\mathbf{x}_0$ 的估计值（通常记作 $\hat{\mathbf{x}}_0$ 或 $\tilde{\mathbf{x}}_0$）：
$$
\hat{\mathbf{x}}_0 = \frac{\mathbf{x}_t - \sigma_t \mathbf{\epsilon}_\theta(\mathbf{x}_t, t)}{\alpha_t}
$$

---

### 3. 三者之间的全转换公式表

在写代码或推导 DPM-Solver++ 时，你可以根据需要随时切换：

| 转换目标       | 基于 噪声 $\epsilon_\theta$                          | 基于 Score $\nabla \ln p_t$                     |
|----------------|------------------------------------------------------|-------------------------------------------------|
| **预测 Score** | $-\frac{\mathbf{\epsilon}_\theta}{\sigma_t}$       | $\nabla \ln p_t$                               |
| **预测 噪声** | $\mathbf{\epsilon}_\theta$                         | $-\sigma_t \nabla \ln p_t$                     |
| **预测 $x_0$**| $\frac{\mathbf{x}_t - \sigma_t \mathbf{\epsilon}_\theta}{\alpha_t}$ | $\frac{\mathbf{x}_t + \sigma_t^2 \nabla \ln p_t}{\alpha_t}$ |

---

### 4. 为什么 DPM-Solver++ 喜欢用 $x_0$ 而不是 Score？

虽然模型底层训练的可能是 Score 或 $\epsilon$，但 DPM-Solver++ 在采样阶段强制将其转换为 $\hat{\mathbf{x}}_0$ 形式，原因有二：

1. **数值稳定性**：
   在 $t \to 0$ 时，$\sigma_t$ 趋于 0。如果你使用 Score 公式（分母有 $\sigma_t$），计算会爆炸。而使用 $x_0$ 公式，由于 $\alpha_t$ 趋于 1，数值非常稳定。

2. **二阶近似更准**：
   DPM-Solver++ 假设在一步之内，原始数据 $x_0$ 是相对恒定的（或者线性变化的）。相比于剧烈抖动的噪声 $\epsilon$，拟合 $x_0$ 的变化曲线带来的离散化误差更小。

---

### 总结

当你看到模型输出一个 output 时，你只需要知道当前的 $\alpha_t$ 和 $\sigma_t$（这些是预设的调度参数），就可以通过上面的加减乘除，把它瞬间变成你需要的任何一种形式。

### 通过 Score Function 直接计算原始数据预测值 ($\hat{\mathbf{x}}_0$)

#### 核心公式
$$
\hat{\mathbf{x}}_0 = \frac{\mathbf{x}_t + \sigma_t^2 \nabla_{\mathbf{x}_t} \ln p_t(\mathbf{x}_t)}{\alpha_t}
$$

#### 推导过程简述（三步到位）

1. **定义前向关系**：
   $$
   \mathbf{x}_t = \alpha_t \mathbf{x}_0 + \sigma_t \mathbf{\epsilon}
   $$

2. **建立 Score 与噪声的关系**：
   已知
   $$
   \nabla_{\mathbf{x}_t} \ln p_t(\mathbf{x}_t) = -\frac{\mathbf{\epsilon}}{\sigma_t}
   $$

3. **代换并移项**：
   将 $\mathbf{\epsilon} = -\sigma_t \nabla_{\mathbf{x}_t} \ln p_t$ 代入前向公式：
   $$
   \mathbf{x}_t = \alpha_t \hat{\mathbf{x}}_0 + \sigma_t (-\sigma_t \nabla_{\mathbf{x}_t} \ln p_t)
   $$

   整理得：
   $$
   \mathbf{x}_t = \alpha_t \hat{\mathbf{x}}_0 - \sigma_t^2 \nabla_{\mathbf{x}_t} \ln p_t
   $$

   最终解出 $\hat{\mathbf{x}}_0$：
   $$
   \hat{\mathbf{x}}_0 = \frac{\mathbf{x}_t + \sigma_t^2 \nabla_{\mathbf{x}_t} \ln p_t(\mathbf{x}_t)}{\alpha_t}
   $$